In [0]:
class Bronze_results():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        results_schema = '''
                resultId INT NOT NULL,
                raceId INT NOT NULL,
                driverId INT NOT NULL,
                constructorId INT NOT NULL,
                number INT,
                grid INT NOT NULL,
                position INT,
                positionText STRING NOT NULL,
                positionOrder INT NOT NULL,
                points FLOAT NOT NULL,
                laps INT NOT NULL,
                time STRING,
                milliseconds INT,
                fastestLap INT,
                rank INT,
                fastestLapTime STRING,
                fastestLapSpeed STRING,
                statusId INT NOT NULL
            '''
        return results_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000) #does nothing because auto loader does not read row by row streaming
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze results Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('ResultsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-results")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.results')
                                 
                    ) 
        print("Done")
        return sQuery   


In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_results_instance = Bronze_results("results",source)
Squery_Bronze_results =Bronze_results_instance.process()
Squery_Bronze_results.awaitTermination()
print("Successfully bronze-ingestion-results stream in running")
Squery_Bronze_results.stop()